# Tahap 2: Case Representation
**Domain:** Pidana Umum - Pemalsuan (Pasal 263 & 266  KUHP)  

Notebook ini mencakup:
1. Load hasil preprocessing dari tahap 1
2. Ekstraksi metadata (no_perkara, tanggal, terdakwa, pasal)
3. Ekstraksi konten kunci (dakwaan, barang bukti)
4. Feature engineering (word count)
5. Verifikasi Sample
6. Simpan ke CSV

## Setup Path

In [1]:
import os

BASE_DIR = r'C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan'

RAW_DIR        = os.path.join(BASE_DIR, 'data', 'raw')
AMAR_DIR       = os.path.join(BASE_DIR, 'data', 'amar')
PROCESSED_DIR  = os.path.join(BASE_DIR, 'data', 'processed')
INDEX_PATH     = os.path.join(BASE_DIR, 'data', 'index.csv')

os.makedirs(PROCESSED_DIR, exist_ok=True)

## 1. Load Hasil Tahap 1

In [2]:
import pandas as pd

index_df = pd.read_csv(INDEX_PATH)
valid_cases = index_df[index_df['valid'] == True]['case_id'].tolist()
print(f'Dokumen valid dari tahap 1: {len(valid_cases)}')

raw_docs = []
for case_id in valid_cases:
    txt_path = os.path.join(RAW_DIR, f'{case_id}.txt')
    if os.path.exists(txt_path):
        with open(txt_path, 'r', encoding='utf-8') as f:
            text = f.read()
        source_file = index_df[index_df['case_id'] == case_id]['file'].values[0]
        raw_docs.append({
            'case_id': case_id,
            'text': text,
            'source_file': source_file
        })

print(f'Berhasil load: {len(raw_docs)} dokumen')

Dokumen valid dari tahap 1: 36
Berhasil load: 36 dokumen


## 2. Fungsi Ekstraksi Metadata


In [3]:
import re

MONTHS_ID = (
    'januari|februari|maret|april|mei|juni|'
    'juli|agustus|september|oktober|november|nopember|desember'
)

_AKHIR_DAKWAAN = re.compile(
    r'\n[ \t]*(?:'
    r'perbuatan\s+(?:para\s+|ia\s+)?terdakwa\s+(?:\w+\s+){0,5}(?:sebagaimana|diatur|diancam)'
    r'|sebagaimana\s+diatur\s+dan\s+diancam'
    r'|kedua\s*:'
    r'|subsidair\s*:'
    r'|lebih\s+subsidair\s*:'
    r'|menimbang[,\s]'
    r'|saksi\s+\w+[,\s]+(?:di\s+bawah|dibawah)\s+sumpah'
    r')',
    re.IGNORECASE
)

_HEADER_DAKWAAN_KEDUA = re.compile(
    r'(?:atau\s+kedua|dan\s+kedua|kedua)\s*:?[\s\n\r]+'
    r'(bahwa\s+(?:[il]a[,\s]+|mereka[,\s]+|para\s+)?terdakwa\s+[\s\S]{100,20000}?)'
    r'(?=\n[ \t]*(?:perbuatan\s+(?:para\s+)?terdakwa\s+(?:\w+\s+){0,5}(?:sebagaimana|diatur)'
    r'|sebagaimana\s+diatur\s+dan\s+diancam'
    r'|ketiga\s*:'
    r'|subsidair\s*:'
    r'|menimbang[,\s]))',
    re.IGNORECASE
)

_KATA_PALSU = re.compile(
    r'mem(?:alsukan|buat\s+surat\s+palsu|buat\s+secara\s+tidak\s+benar)'
    r'|memalsu(?:kan)?\s+surat'
    r'|membuat\s+secara\s+tidak\s+benar'
    r'|memakai\s+surat\s+(?:palsu|yang\s+dipalsukan)'
    r'|menggunakan\s+surat\s+(?:palsu|yang\s+dipalsukan)'
    r'|surat\s+(?:palsu|yang\s+dipalsukan)'
    r'|dokumen\s+(?:palsu|yang\s+dipalsukan)'
    r'|ijazah\s+palsu'
    r'|sertifikat\s+(?:keterampilan\s+)?palsu'
    r'|buku\s+(?:pelaut|nikah|kir)\s+palsu'
    r'|ktp\s+(?:palsu|elektronik\s+palsu)'
    r'|dipalsukan'
    r'|tidak\s+benar\s+atau\s+memalsu'
    r'|keterangan\s+palsu'
    r'|memasukkan\s+keterangan\s+(?:yang\s+)?palsu'
    r'|pemalsuan\s+surat',
    re.IGNORECASE
)

_BUKAN_PEMALSUAN = re.compile(
    r'pasal\s+(?:362|363|364|365|372|374|378|406)\b'
    r'|mengambil\s+barang\s+sesuatu'
    r'|dimiliki\s+secara\s+melawan\s+hukum',
    re.IGNORECASE
)

def first_match(patterns, text, group=1):
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if match:
            value = match.group(group).strip()
            value = re.sub(r'[\s]+', ' ', value)
            value = re.sub(r'\.{2,}', '', value).strip()
            return value if value else None
    return None


def extract_no_perkara(text):
    return first_match([
        r'p\s*u\s*t\s*u\s*s\s*a\s*n\s*[\n\r\s]*'
        r'(?:no\.?|nomor)\s*[:\s]*'
        r'(\d+\s*/\s*[Pp]id[A-Za-z.\-\s]*/\s*\d{4}\s*/\s*[Pp][Nn][^\n,;]{0,30})',
        r'nomor\s*[:\s]\s*(\d+\s*/\s*[Pp]id[A-Za-z.\-\s]*/\s*\d{4}\s*/\s*[Pp][Nn][^\n,;]{0,30})',
    ], text)


def extract_tanggal(text):
    tanggal_pola = rf'(\d{{1,2}}\s+(?:{MONTHS_ID})\s+\d{{4}})'

    patterns = [
        rf'(?:demikian(?:lah)?|diputuskan)\s+dalam\s+'
        rf'(?:sidang\s+permusyawaratan|rapat\s+(?:permusyawaratan|musyawarah)|musyawarah)\s+majelis\s+hakim'
        rf'[\s\S]{{0,300}}?{tanggal_pola}',
        rf'(?:demikian\s+)?putusan\s+ini\s+dijatuhkan\s+dalam\s+rapat\s+permusyawaratan\s+majelis'
        rf'[\s\S]{{0,300}}?{tanggal_pola}',
    ]

    raw_date = None
    for pattern in patterns:
        matches = list(re.finditer(pattern, text, re.IGNORECASE))
        if matches:
            raw_date = matches[-1].group(1).strip()
            break

    if not raw_date:
        return None

    bulan_map = {
        'januari': '01', 'februari': '02', 'maret': '03', 'april': '04',
        'mei': '05', 'juni': '06', 'juli': '07', 'agustus': '08',
        'september': '09', 'oktober': '10', 'november': '11', 'nopember': '11',
        'desember': '12'
    }
    try:
        parts = raw_date.lower().split()
        if len(parts) == 3:
            return f"{parts[2]}-{bulan_map.get(parts[1], '01')}-{parts[0].zfill(2)}"
    except Exception:
        pass
    return raw_date


def extract_terdakwa(text):
    val = first_match([
        r'nama\s+(?:lengkap\s*)?[:\-]\s*([^\n:]{3,80}?)(?:\s*;|\s*\n)',
        r'nama\s+lengkap[\s.:\-]{0,20}([A-Za-z][\s\S]{3,80}?)(?:;|\n\s*\n|tempat\s+lahir)',
    ], text)
    if val:
        val = re.sub(r'\s+\d+\.\s*$', '', val).strip()
        val = re.sub(r'\s+\d+\s*$', '', val).strip()
    return val


def extract_pasal_dakwaan(text):
    pasals = re.findall(
        r'pasal\s+(?:263|264|266)'
        r'(?:\s+ayat\s*\([^)]+\))?'
        r'(?:\s+(?:jo\.?|dan|juncto)\s+pasal\s+\d+(?:\s+ayat\s*\([^)]+\))?)*',
        text,
        re.IGNORECASE
    )
    cleaned = []
    seen = set()
    for p in pasals:
        p = re.sub(r'\(\s*(\d+)\s*\)', r'(\1)', p)
        p = re.sub(r'\s+', ' ', p.strip().lower())
        if p not in seen:
            seen.add(p)
            cleaned.append(p)
    return '; '.join(cleaned[:5]) if cleaned else None


def infer_pasal(amar_text, pasal_dakwaan_str):
    if not amar_text:
        return 'tidak diketahui'
    t = amar_text.lower()

    if 'tidak terbukti' in t or 'membebaskan' in t:
        return 'bebas'

    m = re.search(r'pasal\s+(263|264|266)(?:\s+ayat\s*\((\d+)\))?', t)
    if m:
        pasal = m.group(1)
        ayat = m.group(2)
        return f'pasal {pasal} ayat ({ayat})' if ayat else f'pasal {pasal}'

    is_akta = bool(re.search(r'akta\s+(?:otentik|autentik)|otentik|autentik', t))
    is_memakai = bool(re.search(r'memakai|menggunakan', t))
    is_membuat = bool(re.search(
        r'membuat|memalsukan|memasukkan\s+keterangan\s+palsu'
        r'|pemalsuan\s+surat|pemalsuan\s+akta'
        r'|turut\s+serta\s+(?:dalam\s+)?(?:pemalsuan|membuat|memalsukan)'
        r'|membantu\s+(?:melakukan\s+)?pemalsuan'
        r'|bersama.sama\s+(?:melakukan\s+)?pemalsuan', t))

    if is_akta and is_memakai and not is_membuat:
        return 'pasal 266 ayat (2)'
    if is_akta:
        return 'pasal 266 ayat (1)'
    if is_memakai and not is_membuat:
        return 'pasal 263 ayat (2)'
    if is_membuat:
        if pasal_dakwaan_str and '263' in pasal_dakwaan_str:
            if 'ayat (1)' in pasal_dakwaan_str:
                return 'pasal 263 ayat (1)'
            if 'ayat (2)' in pasal_dakwaan_str:
                return 'pasal 263 ayat (2)'
        return 'pasal 263 ayat (1)'

    if pasal_dakwaan_str:
        d = pasal_dakwaan_str.lower()
        if '266' in d:
            if 'ayat (1)' in d:
                return 'pasal 266 ayat (1)'
            return 'pasal 266'
        if '264' in d:
            return 'pasal 264 ayat (1)'
        if 'ayat (1)' in d and '263' in d:
            return 'pasal 263 ayat (1)'
        if 'ayat (2)' in d and '263' in d:
            return 'pasal 263 ayat (2)'

    return 'tidak diketahui'


def clean_and_sort_pasal(pasal_raw_text):
    if not pasal_raw_text or pd.isna(pasal_raw_text):
        return "[NONE]"

    pasal_found = re.findall(
        r'pasal\s+\d+(?:\s+ayat\s+\(\d+\))?(?:\s+jo\.?\s+pasal\s+\d+(?:\s+ayat\s+\(\d+\))?)?',
        pasal_raw_text.lower()
    )

    unique_pasal = sorted(list(set([p.strip() for p in pasal_found])))

    return "; ".join(unique_pasal) if unique_pasal else "[NONE]"


def extract_dakwaan(text: str, nama_terdakwa: str = '') -> str | None:
    keywords = [
        'primer :', 'primer:',
        'kesatu\n', 'kesatu :', 'kesatu:',
        'pertama :', 'pertama:',
        'tunggal :', 'tunggal:',
        'dakwaan :', 'dakwaan:',
    ]

    text_lower = text.lower()
    idx_start = -1

    for keyword in keywords:
        pos = 0
        while True:
            idx = text_lower.find(keyword, pos)
            if idx < 0:
                break
            window_check = text_lower[idx:idx + 1500]
            if re.search(r'bahwa\s+(?:ia\s+|mereka\s+|para\s+)?terdakwa', window_check):
                idx_start = idx
                break
            pos = idx + 1
        if idx_start >= 0:
            break

    if idx_start < 0:
        nama_kata1 = nama_terdakwa.lower().split()[0] if nama_terdakwa else ''
        for pola in [
            rf'bahwa\s+ia\s+terdakwa\s+{re.escape(nama_kata1)}' if nama_kata1 else None,
            r'bahwa\s+ia\s+terdakwa\b',
            r'bahwa\s+(?:para\s+)?terdakwa\b',
        ]:
            if not pola:
                continue
            m = re.search(pola, text, re.IGNORECASE)
            if m:
                idx_start = max(0, m.start() - 20)
                break

    if idx_start < 0:
        return None

    window = text[idx_start:idx_start + 25000]
    nama_kata1 = nama_terdakwa.lower().split()[0] if nama_terdakwa else ''

    candidates = []

    m_kedua = _HEADER_DAKWAAN_KEDUA.search(window)
    if m_kedua:
        narasi_kedua = m_kedua.group(1)
        m_akhir_kedua = _AKHIR_DAKWAAN.search(narasi_kedua)
        if m_akhir_kedua:
            narasi_kedua = narasi_kedua[:m_akhir_kedua.start()]
        if _KATA_PALSU.search(narasi_kedua) and not _BUKAN_PEMALSUAN.search(narasi_kedua[:1500]):
            candidates.append((10, m_kedua.start(), narasi_kedua))

    for m in re.finditer(
        r'bahwa\s+(?:[il]a[,\s]+|mereka[,\s]+|para\s+)?terdakwa\s+',
        window, re.IGNORECASE
    ):
        narasi = window[m.start():]
        m_akhir = _AKHIR_DAKWAAN.search(narasi)
        if m_akhir:
            narasi = narasi[:m_akhir.start()]

        if not _KATA_PALSU.search(narasi):
            continue

        score = 0
        if nama_kata1:
            if nama_kata1 in narasi.lower()[:300]:
                score += 3
            elif nama_kata1 in narasi.lower():
                score += 1
            else:
                score -= 2
        if re.search(r'dengan\s+cara\s+(?:sebagai\s+berikut|sbb)', narasi, re.IGNORECASE):
            score += 1
        if _BUKAN_PEMALSUAN.search(narasi[:1500]):
            score -= 3

        candidates.append((score, m.start(), narasi))

    if not candidates:
        return None

    candidates.sort(key=lambda x: (-x[0], x[1]))
    best = candidates[0][2]
    return re.sub(r'\s+', ' ', best).strip()[:2000]


def extract_barang_bukti(amar_text):
    match = re.search(
        r'barang\s+bukti\s*(?:berupa)?\s*:?\s*'
        r'([\s\S]+?)'
        r'(?=\n\s*\d{1,2}\s*[\.\)]\s*(?:mem|menetapkan\s+terdakwa\s+membayar)|\Z)',
        amar_text, re.IGNORECASE
    )
    if not match:
        return None
    lines = [l.strip() for l in match.group(1).split('\n') if l.strip()]
    return ' | '.join(lines)[:8000]


def extract_amar(amar_text):
    if not amar_text:
        return None

    blok = re.sub(r'\s+', ' ', amar_text).lower().strip()

    if re.search(r'melepaskan\s+terdakwa|lepas\s+dari\s+(?:segala\s+)?tuntutan', blok):
        return "lepas"
    if re.search(r'tidak\s+terbukti\s+secara\s+sah|membebaskan\s+terdakwa', blok):
        return "bebas"
    if re.search(
        r'(?<!tidak\s)terbukti\s+secara\s+sah\s+dan\s+(?:meyakinkan|menyakinkan)\s+bersalah'
        r'|(?<!tidak\s)terbukti\s+bersalah'
        r'|(?<!tidak\s)terbukti\s+secara\s+sah\s+dan\s+(?:meyakinkan|menyakinkan)\s+melakukan',
        blok
    ):
        return "bersalah"

    return None


def extract_durasi_hukuman(amar_text):
    if not amar_text:
        return None

    text = re.sub(r'\s+', ' ', amar_text.lower())

    pattern_tahun_bulan = r'selama\s*(\d+)\s*[\(\w\s\)]*tahun\s+dan\s+(\d+)\s*[\(\w\s\)]*bulan'
    pattern_tahun_bulan2 = r'selama\s*(\d+)\s*[\(\w\s\)]*tahun\s+(\d+)\s*[\(\w\s\)]*bulan'
    pattern_tahun = r'selama\s*(\d+)\s*[\(\w\s\)]*tahun'
    pattern_bulan = r'selama\s*(\d+)\s*[\(\w\s\)]*bulan'

    m = re.search(pattern_tahun_bulan, text)
    if m:
        return int(m.group(1)) * 12 + int(m.group(2))

    m = re.search(pattern_tahun_bulan2, text)
    if m:
        return int(m.group(1)) * 12 + int(m.group(2))

    m = re.search(pattern_tahun, text)
    if m:
        return int(m.group(1)) * 12

    m = re.search(pattern_bulan, text)
    if m:
        return int(m.group(1))

    return None


def label_hukuman(row):
    if row['amar_putusan'] == 'bebas':
        return 'bebas'
    if row['durasi_bulan'] is None or pd.isna(row['durasi_bulan']):
        return 'tidak tercatat'
    elif row['durasi_bulan'] <= 12:
        return 'ringan'
    else:
        return 'berat'

## 3. Pipeline Ekstraksi

In [4]:
records = []

for fname in sorted(f for f in os.listdir(RAW_DIR) if f.endswith('.txt')):
    case_id = fname.replace('.txt', '')

    if case_id not in valid_cases:
        continue

    with open(os.path.join(RAW_DIR, fname), 'r', encoding='utf-8') as f:
        raw_text = f.read()

    amar_path = os.path.join(AMAR_DIR, fname)
    amar_text = ''
    if os.path.exists(amar_path):
        with open(amar_path, 'r', encoding='utf-8') as f:
            amar_text = f.read()

    terdakwa = extract_terdakwa(raw_text)
    pasal_raw = extract_pasal_dakwaan(raw_text)

    amar_result = extract_amar(amar_text)
    durasi = extract_durasi_hukuman(amar_text)

    record = {
        'case_id'            : case_id,
        'source_file'        : fname,
        'jenis_perkara'      : 'pidana umum - pemalsuan',
        'no_perkara'         : extract_no_perkara(raw_text),
        'tanggal'            : extract_tanggal(raw_text),
        'terdakwa'           : terdakwa,
        'pasal_dakwaan'      : clean_and_sort_pasal(pasal_raw),
        'pasal_terbukti'     : infer_pasal(amar_text, clean_and_sort_pasal(pasal_raw)),
        'dakwaan'            : extract_dakwaan(raw_text, nama_terdakwa=terdakwa),
        'barang_bukti'       : extract_barang_bukti(amar_text),
        'amar_putusan'       : amar_result,
        'durasi_bulan'       : durasi,
        'hukuman'            : label_hukuman({'amar_putusan': amar_result, 'durasi_bulan': durasi}),
        'word_count'         : len(raw_text.split()),
        'text_full'          : raw_text,
    }
    records.append(record)

df = pd.DataFrame(records)

## 4. Audit Hasil Ekstraksi


In [5]:
METADATA_FIELDS = [
    'no_perkara',
    'tanggal',
    'terdakwa',
    'pasal_dakwaan',
    'dakwaan',
    'barang_bukti',
    'amar_putusan'
]

print('=== Coverage Ekstraksi Metadata ===')
for field in METADATA_FIELDS:
    filled = df[field].notna().sum()
    pct = filled / len(df) * 100
    status = 'OK' if pct >= 70 else 'PERLU CEK'
    print(f'  {field:<20}: {filled:>2}/{len(df)} ({pct:.0f}%) [{status}]')

=== Coverage Ekstraksi Metadata ===
  no_perkara          : 36/36 (100%) [OK]
  tanggal             : 36/36 (100%) [OK]
  terdakwa            : 36/36 (100%) [OK]
  pasal_dakwaan       : 36/36 (100%) [OK]
  dakwaan             : 36/36 (100%) [OK]
  barang_bukti        : 36/36 (100%) [OK]
  amar_putusan        : 36/36 (100%) [OK]


In [6]:
df['missing_count'] = df[METADATA_FIELDS].isnull().sum(axis=1)
problematic = df[df['missing_count'] >= 3][['case_id', 'source_file', 'missing_count']]

if len(problematic) > 0:
    print(f'Dokumen dengan 3+ field kosong ({len(problematic)} dokumen):')
    print(problematic.to_string(index=False))
    print('\n=> Cek manual file txt-nya untuk verifikasi.')
else:
    print('Semua dokumen punya minimal 5 field terisi.')

Semua dokumen punya minimal 5 field terisi.


## 5. Verifikasi Sampel Manual

In [ ]:
for _, row in df.head(5).iterrows():
    print(f"\n{'='*60}")
    print(f"case_id       : {row['case_id']}")
    print(f"no_perkara    : {row['no_perkara']}")
    print(f"tanggal       : {row['tanggal']}")
    print(f"terdakwa      : {row['terdakwa']}")
    print(f"pasal_dakwaan : {row['pasal_dakwaan']}")
    print(f"dakwaan       : {str(row['dakwaan'])[:150]}...")
    print(f"barang_bukti  : {str(row['barang_bukti'])[:150]}...")
    print(f"amar_putusan  : {str(row['amar_putusan']) if row['amar_putusan'] else 'None'}")
    print(f"word_count    : {row['word_count']}")


case_id       : case_001
no_perkara    : 1014/pid.b/2019/pn jkt.utr
tanggal       : 2019-10-29
terdakwa      : eka kurniawan
pasal_dakwaan : pasal 263 ayat (2); pasal 266 ayat (1)
dakwaan       : bahwa terdakwa eka kurniawan bersama-sama dengan terdakwa restiana binti rosano boyke pada hari senin tanggal 21 mei 2018 sekitar jam 14.00 wib atau s...
barang_bukti  : 1 (satu) pasang buku nikah palsu nomor 404/089/v/2018 atas nama eka kurniawan (duda) dan restiani (janda), dirampas untuk dimusnahkan dan 1 (satu) buk...
amar_putusan  : bersalah
word_count    : 4741

case_id       : case_002
no_perkara    : 1028/pid.b/2020/pn jkt.utr
tanggal       : 2020-10-23
terdakwa      : gad jered makanoneng als gatot bin rein makanoneng alm
pasal_dakwaan : pasal 263 ayat (1); pasal 263 ayat (2)
dakwaan       : bahwa terdakwa diajukan ke persidangan oleh penuntut umum didakwa berdasarkan surat dakwaan sebagai berikut: bahwa terdakwa gad jered makanoneng als g...
barang_bukti  : 1. 1 (satu) unit handphon

## 6. Simpan ke CSV

In [8]:
COLUMN_ORDER = [
    'case_id', 'no_perkara', 'tanggal', 'jenis_perkara',
    'terdakwa', 'pasal_dakwaan', 'pasal_terbukti',
    'dakwaan', 'barang_bukti', 'amar_putusan',
    'durasi_bulan', 'hukuman',
    'word_count', 'source_file', 'text_full'
]

df_save = df[COLUMN_ORDER].copy()

CSV_PATH = os.path.join(PROCESSED_DIR, 'cases.csv')
df_save.to_csv(CSV_PATH, index=False, encoding='utf-8')
print(f'CSV disimpan: {CSV_PATH}')
print(f'Shape: {df_save.shape}')

PREVIEW_PATH = os.path.join(PROCESSED_DIR, 'cases_preview.csv')
df_save.drop(columns=['text_full']).to_csv(PREVIEW_PATH, index=False, encoding='utf-8')
print(f'Preview CSV (tanpa text_full): {PREVIEW_PATH}')

CSV disimpan: C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\data\processed\cases.csv
Shape: (36, 15)
Preview CSV (tanpa text_full): C:\Users\CENTRALGALAXY\OneDrive\Dokumen\PK\CBR_Pemalsuan\data\processed\cases_preview.csv
